In [1]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import html

print("Fetching CSV and JSONL datasets...")
url_ic = "https://huggingface.co/datasets/mehmetdavut/RubyCraft-3.4-Eval-Logs/resolve/main/ic_before_after.csv"
url_ec = "https://huggingface.co/datasets/mehmetdavut/RubyCraft-3.4-Eval-Logs/resolve/main/ec_before_after.csv"
url_jsonl = "https://huggingface.co/datasets/mehmetdavut/RubyCraft-3.4-Eval-Logs/resolve/main/eval_ec_custom.jsonl"

df_ic = pd.read_csv(url_ic)
df_ec = pd.read_csv(url_ec)

try:
    df_jsonl = pd.read_json(url_jsonl, lines=True)
    print("JSONL solutions loaded successfully! Total rows:", len(df_jsonl))
except Exception as e:
    print("Could not load JSONL. Error:", e)
    df_jsonl = pd.DataFrame()

ec_col = "after_style_mean"
if ec_col in df_ec.columns:
    df_ec["EC_Score"] = df_ec[ec_col]
    if df_ec["EC_Score"].max() > 10.0:
        df_ec["EC_Score"] = df_ec["EC_Score"] / 10
else:
    df_ec["EC_Score"] = 0

print("Data fetch operations completed!")

Fetching CSV and JSONL datasets...
JSONL solutions loaded successfully! Total rows: 6560
Data fetch operations completed!


In [2]:
# Merge IC and EC
df_summary = pd.merge(
    df_ic, df_ec[["model", "EC_Score"]], on="model", how="left"
).rename(columns={"after_pct": "IC_Score"})

# Calculate questions solved
df_summary["Questions_Solved"] = (
    (df_summary["IC_Score"] * 161 / 100).round().astype(int, errors="ignore")
)


# Tagging functions
def categorize_model(name):
    name = name.lower()
    if any(t in name for t in ["gpt", "claude", "oss", "teacher"]):
        return "1. Teacher"
    elif "base" in name:
        return "2. Base (Untrained)"
    elif "lora" in name:
        return "3. LoRA (Trained)"
    return "Other"


def get_base_architecture(name):
    name = name.lower()
    if "qwen" in name and "coder" in name:
        return "Qwen-Coder"
    elif "qwen" in name:
        return "Qwen (Base)"
    elif "llama" in name:
        return "Llama"
    elif "gemma" in name:
        return "Gemma"
    elif "phi" in name:
        return "Phi"
    return "Teacher"


def get_tier(name):
    name = name.lower()
    if any(s in name for s in ["7b", "8b", "9b", "big"]):
        return "Big"
    return "Small"


df_summary["Category"] = df_summary["model"].apply(categorize_model)
df_summary["Architecture"] = df_summary["model"].apply(get_base_architecture)
df_summary["Tier"] = df_summary["model"].apply(get_tier)
df_summary["Model_Group"] = df_summary.apply(
    lambda r: (
        "Teacher"
        if r["Architecture"] == "Teacher"
        else f"{r['Architecture']} ({r['Tier']})"
    ),
    axis=1,
)


# DETECT QUALITY: exp-109 to exp-114 is HQ, others are ALL
def detect_quality(row):
    exp = str(row.get("experiment", ""))
    hq_ids = ["exp-109", "exp-110", "exp-111", "exp-112", "exp-113", "exp-114"]
    if exp in hq_ids or "high_quality" in str(row["model"]).lower():
        return "HQ"
    elif "lora" in str(row["model"]).lower():
        return "ALL"
    return "-"


df_summary["Quality"] = df_summary.apply(detect_quality, axis=1)
model_groups = [
    g
    for g in df_summary["Model_Group"].unique()
    if "Other" not in g and "Teacher" not in g
]
print("Data mapping complete. HQ and ALL categories are now separated.")

Data mapping complete. HQ and ALL categories are now separated.


In [3]:
def generate_detailed_table(selected_group):
    df_filtered = df_summary[
        (df_summary["Model_Group"] == selected_group)
        | (df_summary["Model_Group"] == "Teacher")
    ].copy()

    if df_filtered.empty:
        print("No data found for this model.")
        return None

    columns_to_show = [
        "Category",
        "model",
        "Dataset",
        "Quality",
        "Quantization",
        "IC_Score",
        "Questions_Solved",
        "EC_Score",
    ]

    for col in columns_to_show:
        if col not in df_filtered.columns:
            df_filtered[col] = "-"

    # Sort by Category first, then by IC_Score descending
    display_df = (
        df_filtered[columns_to_show]
        .sort_values(["Category", "IC_Score"], ascending=[True, False])
        .reset_index(drop=True)
    )

    display_df = display_df.rename(
        columns={
            "model": "Model ID",
            "IC_Score": "Logic Pass Rate (%)",
            "Questions_Solved": "Solved (/161)",
            "EC_Score": "Format (/10)",
        }
    )

    def highlight_rows(row):
        if "Teacher" in row["Category"]:
            return [
                "background-color: #fff3cd; color: #856404; font-weight: bold"
            ] * len(row)
        elif "Base" in row["Category"]:
            return ["background-color: #f8f9fa; color: #6c757d"] * len(row)
        elif "LoRA" in row["Category"] and row["Quality"] == "HQ":
            return ["background-color: #d4edda; color: #155724"] * len(row)
        elif "LoRA" in row["Category"]:
            return ["background-color: #e2e3e5; color: #383d41"] * len(row)
        return [""] * len(row)

    (
        display_df.style.apply(highlight_rows, axis=1)
        .format(
            {
                "Logic Pass Rate (%)": "{:.2f}%",
                "Format (/10)": "{:.2f}",
                "Solved (/161)": "{:.0f}",
            }
        )
        .set_caption(
            f"<b>Detailed Results: {selected_group} and Teacher Combinations</b>"
        )
        .set_table_styles(
            [
                {
                    "selector": "caption",
                    "props": [
                        ("font-size", "18px"),
                        ("color", "#2c3e50"),
                        ("margin-bottom", "10px"),
                    ],
                },
                {
                    "selector": "th",
                    "props": [
                        ("font-size", "13px"),
                        ("text-align", "center"),
                        ("background-color", "#34495e"),
                        ("color", "white"),
                    ],
                },
                {
                    "selector": "td",
                    "props": [
                        ("text-align", "center"),
                        ("font-size", "13px"),
                        ("border", "1px solid #dee2e6"),
                    ],
                },
            ]
        )
    )

    # display(styled_table)
    return df_filtered

In [13]:
def generate_code_viewer(df_filtered, task_id):
    if df_jsonl.empty:
        display(HTML("<p>JSONL data not found.</p>"))
        return

    task_col = "task_id" if "task_id" in df_jsonl.columns else "prompt"

    # 1. Identify Models for the 4 Columns
    t_row = df_filtered[df_filtered["Category"].str.contains("Teacher")]
    b_row = df_filtered[df_filtered["Category"].str.contains("Base")]

    # Auto-find BEST ALL and BEST HQ for this specific architecture
    loras = df_filtered[df_filtered["Category"].str.contains("LoRA")]
    all_models = loras[loras["Quality"] == "ALL"]
    hq_models = loras[loras["Quality"] == "HQ"]

    best_all_row = (
        all_models.loc[all_models["IC_Score"].idxmax()]
        if not all_models.empty
        else None
    )
    best_hq_row = (
        hq_models.loc[hq_models["IC_Score"].idxmax()] if not hq_models.empty else None
    )

    def get_solution(row_data, role, quality_target=None):
        m_name = "Unknown"
        e_id = None
        ec_score = "N/A"

        # If there is a record in CSV, get the data
        if row_data is not None:
            m_name = row_data["model"] if isinstance(row_data, pd.Series) else "Unknown"
            e_id = (
                row_data.get("experiment", None)
                if isinstance(row_data, pd.Series)
                else None
            )
            # Handle NaN values for experiment IDs (Crucial for Big Models)
            if pd.isna(e_id) or str(e_id).strip().lower() == "nan":
                e_id = None
            ec_score = (
                f"{row_data['EC_Score']:.1f}/10"
                if isinstance(row_data, pd.Series) and pd.notna(row_data["EC_Score"])
                else "N/A"
            )

        task_data = df_jsonl[df_jsonl[task_col] == task_id]
        if task_data.empty:
            return "Task not found in JSONL.", m_name, "N/A", "N/A"

        match = None
        src_lbl = "Unknown"

        # Role-Based Matching Logic
        if role == "teacher":
            m = task_data[task_data["model_role"].astype(str).str.lower() == "teacher"]
            match = m.iloc[0] if not m.empty else None
            src_lbl = (
                match.get("teacher_model", "Teacher")
                if match is not None
                else "Unknown"
            )

        elif role == "base":
            m = task_data[task_data["model_role"].astype(str).str.lower() == "base"]
            if not m.empty:
                target_clean = str(m_name).lower().replace("_base", "").strip()
                for _, r in m.iterrows():
                    arch = str(r.get("model_architecture", "")).lower()
                    if arch in target_clean or target_clean in arch:
                        match = r
                        break
                if match is None:
                    match = m.iloc[0]
            src_lbl = (
                match.get("model_architecture", "Base")
                if match is not None
                else "Unknown"
            )

        else:  # LoRA (ALL or HQ)
            # 1. Try matching by exact experiment ID if available (e.g. exp-104)
            if e_id and "experiment_id" in task_data.columns:
                m = task_data[task_data["experiment_id"] == e_id]
                match = m.iloc[0] if not m.empty else None

            # 2. SMART FALLBACK FOR BIG MODELS: Search directly by exact model name in JSONL
            if match is None and m_name != "Unknown":
                target_clean = str(m_name).lower().strip()
                if "experiment_id" in task_data.columns:
                    m = task_data[
                        task_data["experiment_id"].astype(str).str.lower()
                        == target_clean
                    ]
                    if not m.empty:
                        match = m.iloc[0]

                if match is None:
                    for _, r in task_data.iterrows():
                        row_str = " ".join([str(v).lower() for v in r.values])
                        if target_clean in row_str:
                            match = r
                            break

            # 3. FORCE SEARCH IN JSONL EVEN IF NOT IN CSV (Compensate for missing CSV rows)
            if match is None and not b_row.empty:
                base_name = str(b_row.iloc[0]["model"]).lower().replace("_base", "")
                for _, r in task_data.iterrows():
                    exp_val = str(r.get("experiment_id", "")).lower()
                    mod_val = str(r.get("model", "")).lower()
                    combined = f"{exp_val} {mod_val}"

                    # If it contains the base model name and the word lora
                    if base_name in combined and "lora" in combined:
                        if quality_target == "ALL" and (
                            "all" in combined
                            or ("hq" not in combined and "high" not in combined)
                        ):
                            match = r
                            m_name = r.get(
                                "experiment_id", "Recovered from JSONL (ALL)"
                            )
                            break
                        elif quality_target == "HQ" and (
                            "hq" in combined or "high" in combined
                        ):
                            match = r
                            m_name = r.get("experiment_id", "Recovered from JSONL (HQ)")
                            break

            src_lbl = str(e_id) if e_id else str(m_name)

        if match is None:
            if row_data is None:
                return "Record Not Found (Training Missing).", "No Data", "N/A", "N/A"
            else:
                return (
                    "Code Not Found in JSONL (Missing Data).",
                    m_name,
                    ec_score,
                    "N/A",
                )

        # Extract and Clean Code
        code = "No Content"
        for c in ["sanitized_response", "raw_response", "completion"]:
            if c in match and pd.notna(match[c]):
                code = str(match[c])
                break

        if task_col == "prompt":
            code = code.replace(task_id, "").strip()
            # Clean up markdown fences safely without breaking the display output
            ruby_fence = "`" * 3 + "ruby"
            code_fence = "`" * 3
            if code.startswith(ruby_fence):
                code = code.replace(ruby_fence, "").replace(code_fence, "").strip()

        return code, m_name, ec_score, src_lbl

    # Fetch all 4 columns
    t_code, t_name, t_ec, t_src = get_solution(
        t_row.iloc[0] if not t_row.empty else None, "teacher"
    )
    b_code, b_name, b_ec, b_src = get_solution(
        b_row.iloc[0] if not b_row.empty else None, "base"
    )
    a_code, a_name, a_ec, a_src = get_solution(best_all_row, "lora", "ALL")
    h_code, h_name, h_ec, h_src = get_solution(best_hq_row, "lora", "HQ")

    html_content = f"""
    <div style="margin-top: 30px;">
        <h3 style="text-align: center; color: #2c3e50;">Task / Prompt Definition</h3>
        <div style="background-color: #f1f3f5; padding: 15px; border-left: 5px solid #3498db; border-radius: 5px; margin-bottom: 25px;">
            <pre style="font-size: 12px; white-space: pre-wrap; font-family: monospace;">{html.escape(str(task_id))}</pre>
        </div>

        <h3 style="text-align: center; color: #2c3e50;">Performance Comparison: Base vs ALL vs HQ</h3>
        <div style="display: flex; gap: 10px; justify-content: center; align-items: stretch;">
            <!-- Column 1: Teacher -->
            <div style="flex: 1; border: 2px solid #f1c40f; border-radius: 8px; padding: 8px; background-color: #fffdf5; display: flex; flex-direction: column;">
                <h4 style="color: #f39c12; text-align: center; margin: 0;">🥇 Teacher</h4>
                <div style="text-align: center; margin: 5px 0;"><span style="background-color: #f39c12; color: white; padding: 2px 8px; border-radius: 12px; font-size: 10px;">EC Score: {t_ec}</span></div>
                <pre style="font-size: 11px; white-space: pre-wrap; flex-grow: 1;"><code>{html.escape(t_code)}</code></pre>
            </div>
            <!-- Column 2: Base -->
            <div style="flex: 1; border: 2px solid #95a5a6; border-radius: 8px; padding: 8px; background-color: #f8f9fa; display: flex; flex-direction: column;">
                <h4 style="color: #7f8c8d; text-align: center; margin: 0;">⚪ Base</h4>
                <div style="text-align: center; margin: 5px 0;"><span style="background-color: #95a5a6; color: white; padding: 2px 8px; border-radius: 12px; font-size: 10px;">EC Score: {b_ec}</span></div>
                <pre style="font-size: 11px; white-space: pre-wrap; flex-grow: 1;"><code>{html.escape(b_code)}</code></pre>
            </div>
            <!-- Column 3: LoRA ALL -->
            <div style="flex: 1; border: 2px solid #3498db; border-radius: 8px; padding: 8px; background-color: #f0f7ff; display: flex; flex-direction: column;">
                <h4 style="color: #2980b9; text-align: center; margin: 0;">📦 LoRA (ALL)</h4>
                <div style="text-align: center; margin: 5px 0;"><span style="background-color: #3498db; color: white; padding: 2px 8px; border-radius: 12px; font-size: 10px;">EC Score: {a_ec}</span></div>
                <pre style="font-size: 11px; white-space: pre-wrap; flex-grow: 1;"><code>{html.escape(a_code)}</code></pre>
                <p style="font-size: 9px; text-align: right; color: #7f8c8d; margin: 0;">Source: {a_src}</p>
            </div>
            <!-- Column 4: LoRA HQ -->
            <div style="flex: 1; border: 2px solid #2ecc71; border-radius: 8px; padding: 8px; background-color: #f4fff8; display: flex; flex-direction: column;">
                <h4 style="color: #27ae60; text-align: center; margin: 0;">🚀 LoRA (HQ)</h4>
                <div style="text-align: center; margin: 5px 0;"><span style="background-color: #2ecc71; color: white; padding: 2px 8px; border-radius: 12px; font-size: 10px;">EC Score: {h_ec}</span></div>
                <pre style="font-size: 11px; white-space: pre-wrap; flex-grow: 1;"><code>{html.escape(h_code)}</code></pre>
                <p style="font-size: 9px; text-align: right; color: #7f8c8d; margin: 0;">Source: {h_src}</p>
            </div>
        </div>
    </div>
    """
    display(HTML(html_content))

In [15]:
# 1. Select Architecture Dropdown
dropdown_group = widgets.Dropdown(
    options=sorted(model_groups),
    description="Architecture:",
    style={"description_width": "initial"},
)

# 2. Select Task Dropdown (Filter for custom40)
task_col = (
    "task_id"
    if "task_id" in df_jsonl.columns
    else ("prompt" if "prompt" in df_jsonl.columns else None)
)
if not df_jsonl.empty and task_col:
    # Safely filter for custom40 if column exists
    if "data_split" in df_jsonl.columns:
        df_custom = df_jsonl[df_jsonl["data_split"] == "custom40"]
        if df_custom.empty:
            df_custom = df_jsonl
    else:
        df_custom = df_jsonl

    unique_tasks = sorted(df_custom[task_col].astype(str).unique().tolist())
    task_options = [((t[:80] + "...") if len(t) > 80 else t, t) for t in unique_tasks]
else:
    task_options = [("N/A", "N/A")]

dropdown_task = widgets.Dropdown(
    options=task_options,
    description="Select Task (Custom40):",
    style={"description_width": "initial"},
)

out_display = widgets.Output()


def update_unified_view(change=None):
    with out_display:
        clear_output(wait=True)
        # 1. Render Table
        df_filt = generate_detailed_table(dropdown_group.value)
        # 2. Render 4-Column Viewer
        if df_filt is not None:
            # We ONLY pass df_filt and task_id. The 4-column engine does the rest!
            generate_code_viewer(df_filt, dropdown_task.value)


# Observers
dropdown_group.observe(update_unified_view, names="value")
dropdown_task.observe(update_unified_view, names="value")

# UI Layout
ui_layout = widgets.VBox([dropdown_group, dropdown_task])
display(ui_layout, out_display)

# Initial render
with out_display:
    update_unified_view()

Output()